# Backlog upload: consolidated CSVs -> Neon

One-time (or occasional) loader for pushing a full historical backlog into every Neon table. Reuses the exact same `db_neon.enforce_schema` / `db_neon.upsert_table` / `db_neon.replace_catalog_tables` helpers as the `scripts/load_*_to_neon.py` scripts — this notebook just runs all six of them, sequentially, against consolidated multi-year CSVs.

**Assumption:** each CSV referenced below already covers your *entire* backlog (every year), not just one month/year. That's what you get if you run `process_orders_to_df.py` / `process_refunds_to_df.py` / `process_labor_to_df.py` / `process_marginedge_purchaises_to_df.py` with their `*_YEAR_DIRS` env var set to a comma-separated list of *every* year folder you have, instead of just one — they already consolidate however many year folders you give them into a single CSV per table.

**Safety notes:**
- Fact tables (orders/checks, items, refunds, labor, MarginEdge purchases) are loaded with `ON CONFLICT ... DO UPDATE` — safe to re-run, rows are upserted by their natural key, nothing gets duplicated.
- Catalog/dimension tables (Toast + MarginEdge catalogs) are loaded with `TRUNCATE + INSERT` (full snapshot replace) — only run those cells with a CSV that represents the *complete current* catalog, not a partial slice.
- Load order between sections doesn't matter for referential integrity.

In [ ]:
import os
import sys
import pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
from sqlalchemy import create_engine

sys.path.append('/content/drive/...../modules')
import db_neon # type: ignore
import schemas # type: ignore

In [ ]:
def load_and_upsert(engine, table_name: str, csv_path: Path, schema: dict, key_columns: list[str]) -> int:
    if not csv_path.exists():
        print(f"[{table_name}] SKIPPED: {csv_path} not found.")
        return 0
    df = pd.read_csv(csv_path)
    df = db_neon.enforce_schema(df, schema)

    # # A backlog CSV built by concatenating multiple extraction runs can contain the same
    # # key more than once (e.g. two year folders covering an overlapping month). Postgres's
    # # ON CONFLICT DO UPDATE can't resolve two incoming rows targeting the same key within
    # # one INSERT (CardinalityViolation), so dedupe before handing off to upsert_table.
    # dupe_count = df.duplicated(subset=key_columns).sum()
    # if dupe_count:
    #     print(f"[{table_name}] WARNING: dropping {dupe_count} duplicate row(s) on {key_columns} "
    #           f"(keeping the last occurrence of each) before upsert.")
    #     df = df.drop_duplicates(subset=key_columns, keep="last")

    return db_neon.upsert_table(engine=engine, table_name=table_name, df=df, key_columns=key_columns)


def load_and_replace(engine, catalog_dir: Path, table_map: dict) -> dict:
    table_to_df = {}
    for table_name, config in table_map.items():
        csv_path = catalog_dir / config["csv"]
        if not csv_path.exists():
            print(f"[{table_name}] SKIPPED: {csv_path} not found.")
            continue
        df = pd.read_csv(csv_path)
        table_to_df[table_name] = db_neon.enforce_schema(df, config["schema"])
    return db_neon.replace_catalog_tables(engine, table_to_df)

In [ ]:
print("Getting Neon Connection:")
neon_connection_string = userdata.get('NEON_CONNECTION_STRING').strip()
engine = create_engine(neon_connection_string)

In [ ]:
ROOT = Path('/content/drive/...../')
BACKLOG_DIRS = {
                "orders": ROOT / "raw_orders", # toast_order_checks.csv, item_sales.csv
                "refunds": ROOT / "raw_refunds", # refunds_daily_summary.csv
                "labor": ROOT / "raw_labor", # labor_time_entries.csv
                "marginedge": ROOT / "raw_purchaise_orders", # marginedge_orders.csv, marginedge_line_items.csv
                "toast_catalogs": ROOT / "raw_catalogs", # catalog_*.csv
                "marginedge_catalogs": ROOT / "raw_catalogs_marginedge", # catalog_categories.csv, catalog_vendors.csv
                "open_meteo": ROOT / "raw_meteo", # open_meteo_summary.csv
                }

for name, path in BACKLOG_DIRS.items():
    print(f"{name} -> {path}  (exists: {path.exists()})")


## Toast Order Data

In [ ]:
load_and_upsert(engine,
                "toast_order_checks",
                BACKLOG_DIRS["orders"] / "toast_order_checks.csv",
                schemas.TOAST_ORDER_CHECKS,
                ["check_guid"])

In [ ]:
load_and_upsert(engine,
                "item_sales",
                BACKLOG_DIRS["orders"] / "item_sales.csv",
                schemas.TOAST_ITEM_SALES,
                ["selection_guid"])

## Toast refunds (`refunds_daily_summary`)
Upsert, keyed on `guid`.


In [ ]:
load_and_upsert(engine,
                "refunds_daily_summary",
                BACKLOG_DIRS["refunds"] / "refunds_daily_summary.csv",
                schemas.TOAST_REFUND_DAILY_SUMMARY,
                ["guid"])

## Toast labor (`labor_time_entries`)
Upsert, keyed on `time_entry_guid`.

In [ ]:
load_and_upsert(engine,
                "labor_time_entries",
                BACKLOG_DIRS["labor"] / "labor_summary.csv",
                schemas.TOAST_LABOR_TIME_ENTRIES, ["time_entry_guid"])

## MarginEdge purchases (`marginedge_orders`, `marginedge_line_items`)
Upsert, keyed on `order_id` / `(order_id, line_item_index)`.

In [ ]:
load_and_upsert(engine, "marginedge_orders",
                BACKLOG_DIRS["marginedge"] / "marginedge_purchaise_orders.csv",
                schemas.MARGINEDGE_ORDERS_SCHEMA,
                ["order_id"])

In [ ]:
load_and_upsert(engine,
                "marginedge_line_items",
                BACKLOG_DIRS["marginedge"] / "marginedge_line_items.csv",
                schemas.MARGINEDGE_LINE_ITEMS_SCHEMA,
                ["order_id", "line_item_index"])

## Toast catalogs (full snapshot replace)
`catalog_tables`, `catalog_dining_options`, `catalog_menu_items`, `catalog_revenue_centers`, `catalog_sales_categories`.

In [ ]:
TOAST_CATALOG_MAP = {
                    "catalog_tables": {"csv": "catalog_tables.csv",
                                       "schema": schemas.TOAST_CATALOG_TABLES},
                    "catalog_dining_options": {"csv": "catalog_dining_options.csv",
                                               "schema": schemas.TOAST_CATALOG_DINING_OPTIONS},
                    "catalog_menu_items": {"csv": "catalog_menu_items.csv",
                                           "schema": schemas.TOAST_CATALOG_MENU_ITEMS},
                    "catalog_revenue_centers": {"csv": "catalog_revenue_centers.csv",
                                                "schema": schemas.TOAST_CATALOG_REVENUE_CENTERS},
                    "catalog_sales_categories": {"csv": "catalog_sales_categories.csv",
                                                 "schema": schemas.TOAST_CATALOG_SALES_CATEGORIES},
                    }

results = load_and_replace(engine,
                           BACKLOG_DIRS["toast_catalogs"],
                           TOAST_CATALOG_MAP)
print(results)

## MarginEdge catalogs (full snapshot replace)
`catalog_purchase_categories`, `catalog_vendors`.

In [ ]:
MARGINEDGE_CATALOGS_MAP = {"catalog_purchase_categories": {"csv": "catalog_categories.csv",
                                                           "schema": schemas.MARGINEDGE_CATALOG_PURCHAISE_CATEGORIES},
                            "catalog_vendors": {"csv": "catalog_vendors.csv",
                                                "schema": schemas.MARGINEDGE_CATALOG_VENDORS_SCHEMA},
                           }

results = load_and_replace(engine,
                           catalog_dir=BACKLOG_DIRS["marginedge_catalogs"],
                           table_map=MARGINEDGE_CATALOGS_MAP)
print(results)

## Weather Data (`weather_daily_summary`)
Upsert, keyed on `yyyyMMdd`.

In [ ]:
load_and_upsert(engine,
                "weather_daily_summary",
                BACKLOG_DIRS["open_meteo"] / "open_meteo_summary.csv",
                schemas.WEATHER_DAILY_SUMMARY,
                ["yyyymmdd"])